# Ejecutor de Colab

Notebook para que los integrantes **sin GPU** aporten cómputo desde Google Colab (issue 006). Hace, en orden:

1. Monta la carpeta compartida de Google Drive.
2. Clona el repositorio en un **commit fijo** (no se copia código en celdas).
3. Instala las dependencias fijadas en `requirements.txt` **sin reinstalar PyTorch** (se usa el de Colab).
4. Verifica la GPU.
5. Verifica la huella (MD5 y tamaño) de los tokenizadores y los datos tokenizados de Drive, y se detiene si no coinciden.
6. Lanza un fragmento de búsqueda con `buscar.py` (o un entrenamiento: pendiente del issue 009).
7. Guarda los resultados **directamente en Drive**: la caché JSON se reescribe tras cada evaluación, así que si Colab corta la sesión basta con volver a ejecutar todo el notebook y continúa sin repetir evaluaciones.

**Uso en Colab:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*, editar solo la celda de configuración y ejecutar todo (*Entorno de ejecución → Ejecutar todas*).

**Modo local:** fuera de Colab, el notebook simula Drive en una carpeta temporal, clona el repositorio local en su `HEAD` y crea artefactos simulados. Sirve para probar el flujo con `APTITUD = "falsa"` sin GPU.

> **Azure:** descartado. Las cuentas *Azure for Students* del equipo tienen cuota 0 de GPU utilizable (ver `issues/done/002-verificar-cuota-gpu-azure.md`), así que no hay versión para VM de Azure.

## 0. Configuración

Lo único que se edita. El resto de celdas no necesita cambios.

In [12]:
import sys

# Modo: "colab" en Google Colab; "local" simula Drive en una carpeta temporal (pruebas sin GPU).
MODO = "colab" if "google.colab" in sys.modules else "local"

# Carpeta compartida de Drive (la que compartió P1). Cada cuenta debe añadir un acceso
# directo a ella en su "Mi unidad" (en drive.google.com: clic derecho sobre la carpeta →
# Organizar → Añadir acceso directo). Se localiza por su ID, así que no importa el nombre
# ni dónde se ponga el acceso directo. CARPETA_DRIVE solo hace falta si la resolución por
# ID falla: entonces, la ruta tal como aparece tras montar Drive.
ID_CARPETA_DRIVE = "17TOnMl178b--icGi1UzQdmjtxNqDO5dm"
CARPETA_DRIVE = None  # p. ej. "/content/drive/MyDrive/<nombre de la carpeta>"

# Repositorio y commit fijado (todas las cuentas deben usar el mismo).
URL_REPO = "https://github.com/FROY33/Sistema-de-traduccion-automatica"
COMMIT =  "6577ad77257b3793191fd7d97d174b8a2f78e4b6"
# Qué ejecutar.
TAREA = "busqueda"          # "busqueda" o "entrenamiento" (TODO issue 009)
METODO = "defecto"        # "aleatoria", "mmas" o "defecto"
PAR = "en-la"               # "en-la" o "en-el"
SEMILLA = 0
FRAGMENTO, FRAGMENTOS = 1, 3  # solo para METODO = "aleatoria": cada cuenta, su fragmento
APTITUD = "real"           # "falsa" (sin GPU) o "real" (entrena cada configuración, issue 003)
# Solo con APTITUD = "real": "real" (≈2 épocas, noche 1) o "minimo" (unos pocos lotes, piloto 008).
# Con un valor distinto de "real" los resultados van a resultados_piloto/ y no a resultados/.
PRESUPUESTO_TOKENS = "5710128"
PRESUPUESTO_EVALUACIONES = None # None: las de buscar.py (40 en total, repartidas en fragmentos); p. ej. 6 en el piloto
MODELO_A_ENTRENAR = None    # TODO issue 009: p. ej. "linea_base", "mejor_aleatorio", "mejor_mmas"

INSTALAR_DEPENDENCIAS = True

# Huella esperada de los artefactos: copiar los valores anotados en
# issues/001-regenerar-artefactos-y-entorno.md (sección "Huella de los artefactos en Drive").
HUELLA_ESPERADA = {
    "en-la": {
        "md5_tokenizador": "e5bc5e38fa4a1a88cabd1b9bbbe9290d",
        "md5_datos": "b23bd1603aa496b441d124e09c63aef3",
        "bytes_datos": 82268900,
    },
    "en-el": {
        "md5_tokenizador": "18c754bde5d431d368369f28bfbe95ac",
        "md5_datos": "963ba2e185e9c82dbde68c59ff4f7ceb",
        "bytes_datos": 1515343376,
    },
}

## 1. Montar Drive

En modo local se usa una carpeta temporal en lugar de Drive.

In [13]:
import subprocess
import tempfile
from pathlib import Path

EN_COLAB = MODO == "colab"

if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    CARPETA_REPO = Path("/content/Sistema-de-traduccion-automatica")
    if CARPETA_DRIVE is None:
        # Los accesos directos a carpetas compartidas aparecen montados en
        # /content/drive/.shortcut-targets-by-id/<ID>/<nombre de la carpeta>.
        por_id = Path("/content/drive/.shortcut-targets-by-id") / ID_CARPETA_DRIVE
        if not por_id.is_dir():
            raise FileNotFoundError(
                f"No se encuentra la carpeta compartida (ID {ID_CARPETA_DRIVE}). Añade un acceso directo "
                "a ella en tu Mi unidad (clic derecho → Organizar → Añadir acceso directo) y vuelve a "
                "ejecutar todo, o fija CARPETA_DRIVE a mano en la celda de configuración."
            )
        subcarpetas = [p for p in por_id.iterdir() if p.is_dir()]
        CARPETA_DRIVE = str(subcarpetas[0] if len(subcarpetas) == 1 else por_id)
    if not Path(CARPETA_DRIVE).is_dir():
        raise FileNotFoundError(
            f"No existe {CARPETA_DRIVE}. Revisa CARPETA_DRIVE (¿añadiste el acceso directo a la carpeta compartida en tu Drive?)."
        )
else:
    BASE_LOCAL = Path(tempfile.gettempdir()) / "ejecutor_colab_local"
    CARPETA_DRIVE = str(BASE_LOCAL / "drive")
    CARPETA_REPO = BASE_LOCAL / "repo"
    # Se clona el repositorio local (la carpeta que contiene este notebook) en su HEAD.
    URL_REPO = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True
    ).stdout.strip()
    COMMIT = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=URL_REPO, capture_output=True, text=True, check=True
    ).stdout.strip()
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)

print(f"Modo: {MODO}")
print(f"Drive: {CARPETA_DRIVE}")
print(f"Repositorio: {URL_REPO} @ {COMMIT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modo: colab
Drive: /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática
Repositorio: https://github.com/FROY33/Sistema-de-traduccion-automatica @ 6577ad77257b3793191fd7d97d174b8a2f78e4b6


## 2. Clonar el repositorio en el commit fijo

Si ya estaba clonado (sesión relanzada), solo se actualiza y se vuelve al commit fijado. Después se comprueba que `HEAD` es ese commit y que no hay cambios sin confirmar.

In [14]:
import importlib


def git(*argumentos, carpeta=None):
    subprocess.run(["git", *argumentos], cwd=carpeta, check=True)


if not (CARPETA_REPO / ".git").exists():
    git("clone", "--quiet", URL_REPO, str(CARPETA_REPO))
else:
    git("fetch", "--quiet", "origin", carpeta=CARPETA_REPO)
git("checkout", "--quiet", "--detach", COMMIT, carpeta=CARPETA_REPO)

if str(CARPETA_REPO) not in sys.path:
    sys.path.insert(0, str(CARPETA_REPO))
import traduccion.ejecutor as ejecutor

ejecutor = importlib.reload(ejecutor)  # por si se cambió de commit en la misma sesión

COMMIT_COMPLETO = ejecutor.comprobar_commit(CARPETA_REPO, COMMIT)
print(f"Repositorio en {COMMIT_COMPLETO} (sin cambios sin confirmar)")

Repositorio en 6577ad77257b3793191fd7d97d174b8a2f78e4b6 (sin cambios sin confirmar)


## 3. Instalar dependencias

Se instalan las versiones fijadas de `requirements.txt` **excepto** `torch` (Colab trae el suyo, compilado para su CUDA) y el kernel de Jupyter (`ipykernel`, `jupyter`). La búsqueda corre en un proceso aparte, así que no hace falta reiniciar el entorno aunque cambien versiones ya cargadas. Si la versión de PyTorch difiere de la fijada, se avisa: la versión real queda registrada en la huella de cada resultado.

In [15]:
TEXTO_REQUIREMENTS = (CARPETA_REPO / "requirements.txt").read_text(encoding="utf-8")
DEPENDENCIAS = ejecutor.dependencias_para_colab(TEXTO_REQUIREMENTS)

if EN_COLAB and INSTALAR_DEPENDENCIAS:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *DEPENDENCIAS], check=True)
    print("Dependencias instaladas:", ", ".join(DEPENDENCIAS))
else:
    print("No se instala nada (modo local o INSTALAR_DEPENDENCIAS = False). En Colab se instalaría:")
    print("  pip install " + " ".join(DEPENDENCIAS))

VERSION_TORCH = ejecutor.version_instalada("torch")
AVISO_TORCH = ejecutor.aviso_version_torch(TEXTO_REQUIREMENTS, VERSION_TORCH)
print(f"PyTorch: {VERSION_TORCH}")
if AVISO_TORCH:
    print(f"AVISO: {AVISO_TORCH}")

Dependencias instaladas: datasets==5.2.0, pandas==3.1.1, numpy==2.5.5, scikit-learn==1.9.1, pyarrow==26.0.1, sentencepiece==0.2.3, transformers==5.24.1, sacrebleu==2.6.0, matplotlib==3.10.8, pytest==9.0.3
PyTorch: 2.14.1+cu130
AVISO: PyTorch instalado 2.14.1+cu130 ≠ fijado 2.15.0. Se usa el de Colab; la versión real queda registrada en la huella de cada resultado.


## 4. Verificar la GPU

La aptitud real y el entrenamiento exigen GPU; con la aptitud falsa solo se informa.

In [16]:
GPU = ejecutor.nombre_gpu()
REQUIERE_GPU = APTITUD == "real" or TAREA == "entrenamiento"
print(f"GPU: {GPU or 'ninguna'}")

if GPU is None and REQUIERE_GPU:
    raise RuntimeError(
        "No hay GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU T4, y volver a ejecutar todo."
    )
if GPU is not None and EN_COLAB and "T4" not in GPU:
    print(f"AVISO: la GPU no es una T4 ({GPU}); queda registrada en el registro de la ejecución.")

GPU: Tesla T4


## 5. Verificar la huella de los artefactos

Calcula el MD5 del tokenizador del par, el MD5 de la carpeta de datos tokenizados (rutas relativas y MD5 de cada archivo) y su tamaño en bytes, y **se detiene** si no coinciden con `HUELLA_ESPERADA` (los valores del issue 001). Estructura esperada en Drive: `artefactos/tok_la_8000.model`, `artefactos/tok_el_16000.model`, `artefactos/datos/en_la/`, `artefactos/datos/en_el/`.

En modo local se crean artefactos simulados y se toman sus propios valores como esperados.

In [17]:
RUTAS = ejecutor.RutasEjecutor(Path(CARPETA_DRIVE))
RUTAS.crear()

if not EN_COLAB:
    ejecutor.crear_artefactos_simulados(RUTAS, PAR)
    HUELLA_ESPERADA[PAR] = ejecutor.huella_artefactos(RUTAS, PAR)

USA_ARTEFACTOS = APTITUD == "real" or TAREA == "entrenamiento"
anotada = all(HUELLA_ESPERADA[PAR].get(c) is not None for c in ejecutor.CAMPOS_ARTEFACTOS)

if USA_ARTEFACTOS or anotada:
    HUELLA_ARTEFACTOS = ejecutor.verificar_artefactos(RUTAS, PAR, HUELLA_ESPERADA[PAR])
    print("Artefactos verificados:")
    for campo, valor in HUELLA_ARTEFACTOS.items():
        print(f"  {campo}: {valor}")
else:
    HUELLA_ARTEFACTOS = None
    print("Aptitud falsa y huella sin anotar: no se usan artefactos, se omite la verificación.")

Artefactos verificados:
  par: en-la
  md5_tokenizador: e5bc5e38fa4a1a88cabd1b9bbbe9290d
  md5_datos: b23bd1603aa496b441d124e09c63aef3
  bytes_datos: 82268900


## 6. Lanzar el trabajo

**Búsqueda:** ejecuta `buscar.py` desde el repositorio clonado con `--salida` en Drive. Si la sesión se corta, volver a ejecutar todo el notebook con la misma configuración continúa desde la caché sin repetir evaluaciones.

**Entrenamiento:** pendiente del issue 009.

In [18]:
import time

inicio = time.time()
if TAREA == "busqueda":
    COMANDO = ejecutor.comando_busqueda(
        CARPETA_REPO, RUTAS, METODO, SEMILLA, PAR, APTITUD, FRAGMENTO, FRAGMENTOS,
        presupuesto_tokens=PRESUPUESTO_TOKENS, presupuesto=PRESUPUESTO_EVALUACIONES,
    )
    print("$ " + " ".join(COMANDO) + "\n")
    CODIGO_SALIDA = ejecutor.ejecutar_con_salida(COMANDO, CARPETA_REPO)
elif TAREA == "entrenamiento":
    # TODO issue 009: lanzar el entrenamiento completo de MODELO_A_ENTRENAR con el script
    # de entrenamiento final (aún no existe) y guardar puntos de control en RUTAS.resultados
    # para poder reanudar tras un corte de Colab.
    raise NotImplementedError("TODO issue 009: el entrenamiento completo aún no está implementado.")
else:
    raise ValueError(f"TAREA desconocida: {TAREA!r}")

REGISTRO = ejecutor.registrar_ejecucion(
    RUTAS,
    {
        "tarea": TAREA,
        "metodo": METODO,
        "par": PAR,
        "semilla": SEMILLA,
        "fragmento": FRAGMENTO,
        "fragmentos": FRAGMENTOS,
        "aptitud": APTITUD,
        "presupuesto_tokens": PRESUPUESTO_TOKENS,
        "presupuesto_evaluaciones": PRESUPUESTO_EVALUACIONES,
        "modo": MODO,
        "commit": COMMIT_COMPLETO,
        "gpu": GPU,
        "torch": VERSION_TORCH,
        "huella_artefactos": HUELLA_ARTEFACTOS,
        "comando": COMANDO,
        "codigo_salida": CODIGO_SALIDA,
        "duracion_s": round(time.time() - inicio, 1),
    },
)
print(f"\nRegistro de la ejecución: {REGISTRO}")
if CODIGO_SALIDA != 0:
    raise RuntimeError(f"buscar.py terminó con código {CODIGO_SALIDA}")

$ /usr/bin/python3 /content/Sistema-de-traduccion-automatica/buscar.py --metodo defecto --semilla 0 --par en-la --aptitud real --salida /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática/resultados_piloto --artefactos /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática/artefactos --presupuesto-tokens 5710128

Método: defecto | semilla: 0 | par: en-la | aptitud: real
Presupuesto de tokens por evaluación: 5710128 | resultados en: /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática/resultados_piloto
Resultados: /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática/resultados_piloto/en-la/real/defecto_semilla0.json
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2

## 7. Resultados en Drive

Resume el JSON del fragmento y muestra la huella de un resultado (debe indicar el commit fijado). Después, quien coordine reúne los JSON de las 3 cuentas con `fusionar.py` (ver README, sección 11).

In [19]:
import json

from traduccion import busqueda

ES_PILOTO = APTITUD == "real" and str(PRESUPUESTO_TOKENS).strip().lower() != "real"
CARPETA_RESULTADOS = RUTAS.resultados_piloto if ES_PILOTO else RUTAS.resultados
RUTA_RESULTADOS = busqueda.ruta_resultados(CARPETA_RESULTADOS, PAR, APTITUD, METODO, SEMILLA, FRAGMENTO, FRAGMENTOS)
resultados = json.loads(RUTA_RESULTADOS.read_text(encoding="utf-8"))["resultados"]
mejor = max(resultados.values(), key=lambda r: r["aptitud"])

print(f"Archivo: {RUTA_RESULTADOS}")
print(f"Evaluaciones guardadas: {len(resultados)}")
print(f"Mejor aptitud: {mejor['aptitud']:.4f}")
print("Huella de un resultado:")
print(json.dumps(mejor["huella"], ensure_ascii=False, indent=2))
assert mejor["huella"]["commit"] == COMMIT_COMPLETO, "La huella no indica el commit fijado"

Archivo: /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática/resultados_piloto/en-la/real/defecto_semilla0.json
Evaluaciones guardadas: 1
Mejor aptitud: 0.1117
Huella de un resultado:
{
  "commit": "6577ad77257b3793191fd7d97d174b8a2f78e4b6",
  "arbol_sucio": false,
  "md5_tokenizador": "e5bc5e38fa4a1a88cabd1b9bbbe9290d",
  "tamano_entrenamiento": 78463,
  "gpu": "Tesla T4",
  "python": "3.13.16",
  "plataforma": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "versiones": {
    "torch": "2.14.1+cu130",
    "transformers": "5.24.1",
    "sentencepiece": "0.2.3",
    "sacrebleu": "2.6.0",
    "datasets": "5.2.0"
  },
  "presupuesto_tokens": 5710128,
  "tokens_por_lote": 8192
}


# Fusion

In [20]:
!cd {CARPETA_REPO} && python fusionar.py "{RUTAS.resultados_piloto}/en-la/real/aleatoria_semilla0_fragmento*de3.json" --salida "{RUTAS.resultados_piloto}/en-la/real/aleatoria_semilla0.json"

Archivos: 3 | resultados fusionados: 9
Duplicados descartados: 0
Huella de referencia: {'par': 'en-la', 'tipo_aptitud': 'real', 'commit': '6577ad77257b3793191fd7d97d174b8a2f78e4b6', 'md5_tokenizador': 'e5bc5e38fa4a1a88cabd1b9bbbe9290d', 'tamano_entrenamiento': 78463, 'presupuesto_tokens': 20000}
Fusión escrita en: /content/drive/.shortcut-targets-by-id/17TOnMl178b--icGi1UzQdmjtxNqDO5dm/Sistema de traducción automática/resultados_piloto/en-la/real/aleatoria_semilla0.json


In [21]:
import json
fus = json.loads((RUTAS.resultados_piloto / "en-la/real/aleatoria_semilla0.json").read_text(encoding="utf-8"))["resultados"]
for r in fus.values():
    print(f"{r['estado']:<22} aptitud {r['aptitud']:.4f}  tiempo {r.get('tiempo', 0):6.1f}s  "
          f"parámetros {r.get('parametros') or 0:>12,}  gpu {r['huella'].get('gpu')}")

ok                     aptitud 0.0000  tiempo   27.3s  parámetros   16,087,296  gpu Tesla T4
ok                     aptitud 0.0000  tiempo    7.6s  parámetros   20,818,560  gpu Tesla T4
ok                     aptitud 0.0000  tiempo   10.1s  parámetros    9,164,544  gpu Tesla T4
ok                     aptitud 0.0000  tiempo   21.7s  parámetros   35,622,400  gpu Tesla T4
ok                     aptitud 0.0000  tiempo   10.7s  parámetros   24,073,728  gpu Tesla T4
ok                     aptitud 0.0000  tiempo    7.2s  parámetros    9,946,880  gpu Tesla T4
ok                     aptitud 0.0000  tiempo   19.4s  parámetros   18,816,000  gpu Tesla T4
ok                     aptitud 0.0000  tiempo   11.7s  parámetros   17,279,616  gpu Tesla T4
ok                     aptitud 0.0000  tiempo    8.9s  parámetros   21,970,432  gpu Tesla T4


In [22]:
r = next(iter(fus.values()))
print("chrF:", r["chrf"], "| pérdida final:", r.get("perdida_final"), "| fp16:", r["fp16"])
for e in r["ejemplos"][:3]:
    print("REF:", e["referencia"][:100]); print("HIP:", repr(e["hipotesis"][:100])); print()

chrF: 0.0 | pérdida final: 9.052127838134766 | fp16: True
REF: si autem faenum agri quod hodie est et cras in clibanum mittitur deus sic vestit quanto magis vos mi
HIP: ''

REF: sic ignea lora cum daret et rapido sol natum imponeret axi, gaudentem lacrimans astra insidiosa doce
HIP: ''

REF: et imperator tamen quod non habet, non dat: non potest propagare vitam, potest honestare militiam.
HIP: ''

